# Exemplo 10: Parquet e Delta Lake

**Objetivos:**
- Compreender o formato Parquet (columnar)
- Comparar performance: CSV vs JSON vs Parquet
- Introdução ao Delta Lake (transações ACID)
- Particionamento de dados

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month, dayofmonth
import time

# Criar sessão Spark
spark = SparkSession.builder \
    .appName("Parquet_Delta_Demo") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## 1. Criar dataset de exemplo

In [ ]:
# Criar dados sintéticos (simulando transações financeiras)
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
n_records = 100000

data = {
    'id': range(n_records),
    'data': [datetime(2024, 1, 1) + timedelta(days=int(x)) for x in np.random.randint(0, 365, n_records)],
    'ativo': np.random.choice(['BTC', 'ETH', 'SOL', 'ADA'], n_records),
    'preco': np.random.uniform(100, 50000, n_records),
    'quantidade': np.random.uniform(0.001, 10, n_records),
    'exchange': np.random.choice(['Binance', 'Kraken', 'Coinbase', 'Bitstamp'], n_records)
}

pandas_df = pd.DataFrame(data)
df = spark.createDataFrame(pandas_df)

df.show(5)
print(f"Total de registos: {df.count():,}")

## 2. Comparar formatos: CSV vs JSON vs Parquet

In [ ]:
# Escrever em CSV
start = time.time()
df.write.mode("overwrite").csv("output_csv", header=True)
csv_time = time.time() - start

# Escrever em JSON
start = time.time()
df.write.mode("overwrite").json("output_json")
json_time = time.time() - start

# Escrever em Parquet
start = time.time()
df.write.mode("overwrite").parquet("output_parquet")
parquet_time = time.time() - start

print(f"Tempos de escrita:")
print(f"  CSV:    {csv_time:.2f}s")
print(f"  JSON:   {json_time:.2f}s")
print(f"  Parquet: {parquet_time:.2f}s")

In [ ]:
import os

def get_dir_size(path):
    total = 0
    for entry in os.scandir(path):
        if entry.is_file():
            total += entry.stat().st_size
    return total / (1024*1024)  # MB

print(f"Tamanho dos ficheiros:")
print(f"  CSV:     {get_dir_size('output_csv'):.2f} MB")
print(f"  JSON:    {get_dir_size('output_json'):.2f} MB")
print(f"  Parquet: {get_dir_size('output_parquet'):.2f} MB")

In [ ]:
# Comparar leitura
start = time.time()
df_csv = spark.read.csv("output_csv", header=True, inferSchema=True)
df_csv.count()
csv_read_time = time.time() - start

start = time.time()
df_parquet = spark.read.parquet("output_parquet")
df_parquet.count()
parquet_read_time = time.time() - start

print(f"Tempos de leitura:")
print(f"  CSV:     {csv_read_time:.2f}s")
print(f"  Parquet: {parquet_read_time:.2f}s")
print(f"\nParquet é {csv_read_time/parquet_read_time:.1f}x mais rápido!")

## 3. Vantagens do Parquet

### 3.1 Pushdown de predicados (predicate pushdown)

In [ ]:
# No Parquet, apenas as colunas necessárias são lidas
# Estatísticas por ficheiro permitem skip de blocos

# Ler apenas algumas colunas (column pruning)
df_parquet.select("ativo", "preco").filter(col("ativo") == "BTC").show(5)

# Ver o plano de execução (physical plan mostra Parquet filter pushdown)
df_parquet.filter(col("ativo") == "BTC").explain()

## 4. Particionamento

In [ ]:
# Adicionar colunas de ano e mês
df_part = df.withColumn("ano", year("data"))\
            .withColumn("mes", month("data"))

# Escrever particionado
df_part.write \
    .partitionBy("ano", "mes") \
    .mode("overwrite") \
    .parquet("output_particionado")

print("Dados particionados por ano/mes")

# Ver estrutura de diretórios criada
import subprocess
result = subprocess.run(['tree', '-L', '3', 'output_particionado'], 
                       capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "Instale 'tree' para ver a estrutura")

In [ ]:
# Query em dados particionados (partition pruning)
start = time.time()

df_jan = spark.read.parquet("output_particionado")\
    .filter((col("ano") == 2024) & (col("mes") == 1))

print(f"Registos de Janeiro 2024: {df_jan.count()}")
print(f"Tempo de query: {time.time() - start:.3f}s")

# Spark só lê os ficheiros da partição ano=2024/mes=1
df_jan.explain()

## 5. Delta Lake (opcional)

Delta Lake adiciona transações ACID sobre ficheiros Parquet.

**Requisito:** `pip install delta-spark`

In [ ]:
try:
    from delta import DeltaTable
    
    # Escrever como Delta
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .save("output_delta")
    
    print("Dados escritos em formato Delta")
    
    # Ler Delta
    df_delta = spark.read.format("delta").load("output_delta")
    print(f"Registos lidos: {df_delta.count()}")
    
    # Ver histórico (time travel)
    delta_table = DeltaTable.forPath(spark, "output_delta")
    delta_table.history().show(5)
    
except ImportError:
    print("Delta Lake não instalado. Execute: pip install delta-spark")

## Resumo

| Formato | Tamanho | Velocidade | Funcionalidades |
|---------|---------|------------|-----------------|
| CSV     | Grande  | Lenta      | Universal       |
| JSON    | Grande  | Lenta      | Semi-estruturado|
| Parquet | Pequeno | Rápida     | Columnar, compressão |
| Delta   | Pequeno | Rápida     | ACID, time travel |

**Recomendação:** Use sempre Parquet para datasets > 1GB em pipelines de Big Data.

In [ ]:
# Limpar ficheiros temporários
import shutil
for d in ['output_csv', 'output_json', 'output_parquet', 'output_particionado', 'output_delta']:
    if os.path.exists(d):
        shutil.rmtree(d)
        
print("Cleanup completo!")
spark.stop()